<style>
body, .reveal, .reveal .slides, .reveal .slides section,
.jp-RenderedHTMLCommon, .jp-MarkdownOutput, .rendered_html {
  font-family: Arial, "DejaVu Sans", "Noto Sans", "Liberation Sans", sans-serif !important;
  font-kerning: normal;
}
.reveal h1, .reveal h2, .reveal h3, .reveal h4,
.jp-RenderedHTMLCommon h1, .jp-RenderedHTMLCommon h2,
.jp-RenderedHTMLCommon h3, .jp-RenderedHTMLCommon h4 {
  font-family: Arial, "DejaVu Sans", "Noto Sans", "Liberation Sans", sans-serif !important;
  font-weight: 600;
}
.reveal code, .reveal pre, .jp-RenderedHTMLCommon code, .jp-RenderedHTMLCommon pre {
  font-family: Menlo, Consolas, "DejaVu Sans Mono", "Liberation Mono", monospace !important;
}
</style>

# Python (data science) - Machine Learning - case study

_Mikołaj Leszczuk_

## Konspekt

* Problem PGC/UGC: klasyfikacja treści wideo.
* Przygotowanie danych i jeden podział `train/test`.
* Porównanie: `RandomForestClassifier` i `DecisionTreeClassifier`.
* `accuracy`, `precision`, `recall`, `f1-score` i macierz pomyłek.
* Wizualizacja drzewa oraz ćwiczenia.

## PGC vs UGC

**PGC** (_Professionally-Generated Content_) to treści przygotowane profesjonalnie, np. materiał telewizyjny albo produkcja streamingowa.

**UGC** (_User-Generated Content_) to treści tworzone przez użytkowników, np. nagrania z telefonu lub amatorskie materiały internetowe.

Według Cisco wideo stanowiło większość ruchu internetowego. To uzasadnia automatyczną analizę treści.

<img src="https://raw.githubusercontent.com/miklesz/Courses/main/Machine%20Learning/cisco1.png" style="max-width: 640px; width: 68%; display: block; margin: 10px auto 0;">

Więcej ruchu na użytkownika oznacza więcej materiałów o różnej jakości.

<img src="https://raw.githubusercontent.com/miklesz/Courses/main/Machine%20Learning/cisco2.png" style="max-width: 640px; width: 68%; display: block; margin: 10px auto 0;">

Problem dla modelu: na podstawie wartości wskaźników wizyjnych zdecydować, czy przykład należy do **PGC**, czy do **UGC**.

Będziemy używać zbioru danych wartości [wskaźników wizyjnych](https://qoe.agh.edu.pl/pl/indicators/) oraz dwóch klasyfikatorów: lasu losowego i drzewa decyzyjnego.

### Importowanie bibliotek

Ten fragment wymaga pakietów `scikit-learn`, `numpy` i `matplotlib` w tym samym kernelu Jupyter, w którym uruchamiamy notebook.

Jeśli pojawi się `ModuleNotFoundError`, najpierw uruchom w osobnej komórce:

```python
%pip install scikit-learn numpy matplotlib
```

In [1]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn import tree
from numpy import genfromtxt, unique
import matplotlib.pyplot as plt
from pathlib import Path

### Load dataset

`X` przechowuje cechy wejściowe, czyli liczby opisujące właściwości obrazu/wideo.

`Y` przechowuje etykiety klas: `0` oznacza PGC, a `1` oznacza UGC.

In [2]:
X = genfromtxt('all_DB_data.csv', delimiter=',')
Y = genfromtxt('all_DB_target.csv', delimiter=',')

FEATURE_NAMES = [
    'Blockiness',
    'SA',
    'Blockloss',
    'Blur',
    'TA',
    'Exposure',
    'Contrast',
    'Noise',
    'Slice',
    'Flickering',
]
CLASS_NAMES = ['PGC', 'UGC']

### Spojrzenie na dane

Zbiór ma 10 cech wejściowych i jedną etykietę klasy. Nie drukujemy całej tablicy, tylko kilka pierwszych wierszy, żeby slajd pozostał czytelny.

In [3]:
print(X[:5])

[[0.52379938 0.11750753 0.05500586 0.08249901 0.27094136 0.37067979
  0.41226285 0.07608165 0.17657655 0.11191889]
 [0.37652329 0.4537487  0.00530719 0.07567454 0.24496533 0.31797659
  0.17715422 0.22677971 0.29282914 0.1025995 ]
 [0.58203674 0.58869758 0.0680501  0.06353426 0.33716392 0.23463782
  0.13246105 0.07901491 0.30424831 0.34149041]
 [0.5824949  0.61081225 0.10724562 0.29247241 0.16356031 0.32081625
  0.39029163 0.23555593 0.30851207 0.33094086]
 [0.54660014 0.13207707 0.04557966 0.10102863 0.07393759 0.19311635
  0.1353505  0.13339208 0.24070533 0.24593287]]


#### Funkcje wejściowe

To nazwy kolumn, które model dostaje jako dane wejściowe.

In [4]:
for index, name in enumerate(FEATURE_NAMES, start=1):
    print(f'{index:2}. {name}')

 1. Blockiness
 2. SA
 3. Blockloss
 4. Blur
 5. TA
 6. Exposure
 7. Contrast
 8. Noise
 9. Slice
10. Flickering


#### Etykiety klas

Każda etykieta odpowiada jednemu wierszowi z `X`. Tutaj pokazujemy tylko początek wektora `Y`.

In [11]:
print(Y[:30].astype(int))

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


#### Class balance

Warto sprawdzić proporcje klas. Przy niezbalansowanych danych sama dokładność (`accuracy`) może dawać zbyt optymistyczny obraz modelu.

In [12]:
values, counts = unique(Y.astype(int), return_counts=True)
for value, count in zip(values, counts):
    print(f'{CLASS_NAMES[value]} ({value}): {count}')

PGC (0): 68
UGC (1): 340


##### PGC sample

Treść profesjonalna zwykle jest tworzona w kontrolowanych warunkach i ma bardziej przewidywalną jakość techniczną.

<img src="https://raw.githubusercontent.com/miklesz/Courses/main/Machine%20Learning/full-pgc.png" style="max-width: 880px; width: 90%; display: block; margin: 14px auto 0;">

##### UGC sample

Treść użytkownika bywa bardziej zróżnicowana: inne oświetlenie, ruch kamery, kompresja, szum albo niestabilny kadr.

<img src="https://raw.githubusercontent.com/miklesz/Courses/main/Machine%20Learning/full-ugc.png" style="max-width: 880px; width: 90%; display: block; margin: 14px auto 0;">

#### Wymiary danych

Liczba wierszy w `X` musi zgadzać się z liczbą etykiet w `Y`. Liczba kolumn w `X` to liczba cech wejściowych.

In [13]:
print(f'X: {X.shape[0]} przykładów, {X.shape[1]} cech')
print(f'Y: {Y.shape[0]} etykiet')

X: 408 przykładów, 10 cech
Y: 408 etykiet


### Train/test split (80/20)

Dane dzielimy tylko raz. Zbiór treningowy służy do uczenia modelu, a zbiór testowy zostaje odłożony do sprawdzenia jakości.

`stratify=Y` zachowuje proporcje PGC/UGC w obu częściach.

In [14]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42,
    stratify=Y,
)

print(X_train.shape, Y_train.shape)
print(X_test.shape, Y_test.shape)

(326, 10) (326,)
(82, 10) (82,)


## Random forest

#### Trenowanie modelu

Las losowy buduje wiele drzew decyzyjnych. Każde drzewo głosuje, a wynik końcowy jest decyzją całego zespołu.

In [15]:
forest_clf = RandomForestClassifier(random_state=42)
forest_clf.fit(X_train, Y_train);

#### Znaczenie cech

`feature_importances_` pokazuje, które cechy najczęściej pomagały modelowi w podejmowaniu decyzji. Wyniki sortujemy malejąco, żeby łatwiej je omówić.

In [16]:
forest_importances = sorted(
    zip(FEATURE_NAMES, forest_clf.feature_importances_),
    key=lambda item: item[1],
    reverse=True,
)

for name, importance in forest_importances:
    print(f'{name:12s} {importance:.3f}')

Flickering   0.180
Blockloss    0.171
Slice        0.123
Noise        0.121
TA           0.115
Blur         0.098
Blockiness   0.085
Exposure     0.052
Contrast     0.033
SA           0.023


#### Predykcja na zbiorze testowym

Porównujemy przewidywania modelu z prawdziwymi etykietami. Pokazujemy tylko pierwsze elementy, bo cały wektor jest zbyt długi na slajd.

In [17]:
forest_pred = forest_clf.predict(X_test)
print('Predykcja:', forest_pred[:25].astype(int))
print('Prawda:   ', Y_test[:25].astype(int))

Predykcja: [1 1 1 1 1 1 1 1 0 1 1 1 0 1 1 1 1 1 0 1 1 1 0 1 1]
Prawda:    [1 1 1 1 1 1 1 1 0 1 1 0 0 1 1 1 1 1 0 1 0 1 1 1 1]


#### Accuracy

Dla klasyfikatorów `score()` zwraca dokładność (`accuracy`): odsetek poprawnych odpowiedzi na zbiorze testowym.

In [18]:
forest_accuracy = forest_clf.score(X_test, Y_test)
print(f'accuracy = {forest_accuracy:.3f}')

accuracy = 0.915


## Drzewo decyzyjne

#### Trenowanie modelu

Pojedyncze drzewo decyzyjne jest zwykle bardziej czytelne niż las losowy, ale łatwiej może dopasować się za mocno do szczegółów danych treningowych.

In [19]:
tree_clf = DecisionTreeClassifier(random_state=42)
tree_clf.fit(X_train, Y_train);

#### Znaczenie cech

Dla pojedynczego drzewa znaczenia cech mogą być mniej stabilne niż w lesie losowym, bo wynik zależy od jednej struktury podziałów.

In [20]:
tree_importances = sorted(
    zip(FEATURE_NAMES, tree_clf.feature_importances_),
    key=lambda item: item[1],
    reverse=True,
)

for name, importance in tree_importances:
    print(f'{name:12s} {importance:.3f}')

Flickering   0.169
Noise        0.168
Slice        0.159
Blockloss    0.146
Blur         0.143
TA           0.076
Exposure     0.055
Blockiness   0.051
Contrast     0.033
SA           0.000


#### Predykcja na zbiorze testowym

In [21]:
tree_pred = tree_clf.predict(X_test)
print('Predykcja:', tree_pred[:25].astype(int))
print('Prawda:   ', Y_test[:25].astype(int))

Predykcja: [1 1 1 1 1 1 1 1 0 1 1 1 0 1 1 1 1 1 0 1 1 1 0 1 1]
Prawda:    [1 1 1 1 1 1 1 1 0 1 1 0 0 1 1 1 1 1 0 1 0 1 1 1 1]


#### Accuracy

Wynik drzewa warto porównać z lasem losowym. Na małym lub prostym zbiorze pojedyncze drzewo może czasem wypaść podobnie albo nawet lepiej.

In [22]:
tree_accuracy = tree_clf.score(X_test, Y_test)
print(f'Random forest: {forest_accuracy:.3f}')
print(f'Decision tree: {tree_accuracy:.3f}')

Random forest: 0.915
Decision tree: 0.927


## Precision, recall, f1-score

`accuracy` mówi, jaki procent przykładów sklasyfikowaliśmy poprawnie. Przy nierównych klasach to za mało.

* `precision`: z przewidzianych przykładów danej klasy, ile było naprawdę poprawnych.
* `recall`: z rzeczywistych przykładów danej klasy, ile model odnalazł.
* `f1-score`: średnia harmoniczna `precision` i `recall`.

<img src="assets/ml-alternatives/precision-recall-horizontal.png" style="width: 100%; max-width: 1180px; height: auto; max-height: 92vh; object-fit: contain; display: block; margin: 0 auto;">

`f1-score` rośnie tylko wtedy, gdy zarówno `precision`, jak i `recall` są sensowne.

<img src="https://wikimedia.org/api/rest_v1/media/math/render/svg/dd577aee2dd35c5b0e349327528a5ac606c7bbbf" style="max-width: 460px; width: 58%; display: block; margin: 18px auto 0;">

### Metryki dla drzewa

`average='binary'` liczy metryki dla klasy pozytywnej, tutaj dla etykiety `1`, czyli UGC.

In [23]:
precision, recall, f1, _ = precision_recall_fscore_support(
    Y_test,
    tree_pred,
    average='binary',
    zero_division=0,
)

print(f'precision = {precision:.3f}')
print(f'recall    = {recall:.3f}')
print(f'f1-score  = {f1:.3f}')

precision = 0.931
recall    = 0.985
f1-score  = 0.957


### Raport klasyfikacji

`classification_report` pokazuje metryki osobno dla każdej klasy. To wygodniejsze niż patrzenie tylko na jedną liczbę.

In [24]:
print(classification_report(
    Y_test,
    tree_pred,
    target_names=CLASS_NAMES,
    zero_division=0,
))

              precision    recall  f1-score   support

         PGC       0.90      0.64      0.75        14
         UGC       0.93      0.99      0.96        68

    accuracy                           0.93        82
   macro avg       0.92      0.81      0.85        82
weighted avg       0.93      0.93      0.92        82



### Confusion matrix

Macierz pomyłek pokazuje, ile razy model trafił i jakiego rodzaju błędy popełnił.

Wiersze oznaczają prawdziwe klasy, a kolumny klasy przewidziane przez model.

In [25]:
matrix = confusion_matrix(Y_test, tree_pred)
print(matrix)

[[ 9  5]
 [ 1 67]]


## Wizualizacja

### Reprezentacja tekstowa

`export_text()` zapisuje reguły drzewa jako tekst. Ograniczamy głębokość wydruku, żeby slajd był czytelny.

In [28]:
text_representation = tree.export_text(
    tree_clf,
    feature_names=FEATURE_NAMES,
    max_depth=3,
)
print(text_representation)

|--- Blockloss <= 0.20
|   |--- Flickering <= 0.48
|   |   |--- Noise <= 0.35
|   |   |   |--- Slice <= 0.35
|   |   |   |   |--- truncated branch of depth 9
|   |   |   |--- Slice >  0.35
|   |   |   |   |--- class: 1.0
|   |   |--- Noise >  0.35
|   |   |   |--- class: 1.0
|   |--- Flickering >  0.48
|   |   |--- class: 1.0
|--- Blockloss >  0.20
|   |--- class: 1.0



### Plot tree

`plot_tree()` daje szybką wizualizację bez dodatkowych narzędzi. Kod poniżej zapisuje obraz, a podgląd pokazujemy na następnym slajdzie.

In [ ]:
Path('assets/ml-alternatives').mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(11, 7))
tree.plot_tree(
    tree_clf,
    feature_names=FEATURE_NAMES,
    class_names=CLASS_NAMES,
    filled=True,
    rounded=True,
    max_depth=2,
    fontsize=12,
    impurity=False,
    ax=ax,
)
ax.margins(x=0.01, y=0.02)
fig.tight_layout(pad=0.15)
fig.savefig('assets/ml-alternatives/lesson20-plot-tree.png', dpi=180, bbox_inches='tight', pad_inches=0.05)
plt.close(fig)

<img src="assets/ml-alternatives/lesson20-plot-tree.png" style="height: 650px; max-height: 88vh; width: auto; max-width: 100%; object-fit: contain; display: block; margin: 0 auto;">

### Optional: dtreeviz

`dtreeviz` potrafi wygenerować bardziej dopracowaną wizualizację drzewa, ale wymaga dodatkowego pakietu oraz programu `graphviz` zainstalowanego w systemie.

Dlatego główną ścieżką na zajęciach jest `plot_tree()`, a `dtreeviz` można potraktować jako rozszerzenie dla chętnych.

## Ćwiczenia dodatkowe

### Ćwiczenie 1. Model bazowy

Zbuduj model, który zawsze przewiduje najczęstszą klasę. Porównaj jego `accuracy` z wynikiem drzewa decyzyjnego.

### Rozwiązanie 1

In [ ]:
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, Y_train)

print(f'Baseline accuracy: {baseline.score(X_test, Y_test):.3f}')
print(f'Decision tree accuracy: {tree_accuracy:.3f}')

Jeśli model jest tylko niewiele lepszy od baseline'u, wysoka dokładność może wynikać głównie z przewagi jednej klasy.

### Ćwiczenie 2. Raport klasyfikacji

Wypisz pełny raport klasyfikacji dla drzewa decyzyjnego: `precision`, `recall` i `f1-score`.

### Rozwiązanie 2

In [ ]:
print(classification_report(
    Y_test,
    tree_pred,
    target_names=CLASS_NAMES,
    zero_division=0,
))

### Ćwiczenie 3. Confusion matrix

Policz macierz pomyłek dla lasu losowego. Sprawdź, ile razy model pomylił PGC z UGC.

### Rozwiązanie 3

In [ ]:
confusion_matrix(Y_test, forest_pred)

### Ćwiczenie 4. Max depth

Sprawdź, jak zmienia się wynik drzewa po ograniczeniu parametru `max_depth`.

### Rozwiązanie 4

In [ ]:
for depth in [1, 2, 3, 4, 5, None]:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, Y_train)
    print(f'max_depth={str(depth):>4}: accuracy={model.score(X_test, Y_test):.3f}')